<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/02_PC_delivery_level.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Delivery-level PC causal discovery (per city)

This notebook implements a two-stage causal discovery pipeline to identify factors influencing delivery ETA (eta_mins) across three Chinese cities (Shanghai, Hangzhou, Chongqing). The approach combines:

- Fisher-Z bootstrap for skeleton discovery

- FastKCI validation for nonlinear dependency confirmation

- Background knowledge to enforce domain constraints

Runs the **PC algorithm** separately for **Shanghai**, **Hangzhou**, and **Chongqing** using **FastKCI** as the conditional independence test.

- 1,000 stratified records per city
- 10 parallel bootstraps (joblib, 10 workers)
- One directed consensus DAG per city





Goal :
```
ETA (Target)
│
├── Order Geometry
├── Batch Structure
├── Courier Operational State
├── Environment
├── Temporal Context
└── POI Characteristics
```




In [ ]:
local = False

# Configure SIMD before importing NumPy/SciPy when running locally on AVX-512 hardware.
if local:
    import os
    os.environ.setdefault('MKL_ENABLE_INSTRUCTIONS', 'AVX512')
    os.environ.setdefault('OMP_NUM_THREADS', '1')  # joblib owns parallelism
    os.environ.setdefault('OPENBLAS_CORETYPE', 'SKYLAKEX')
    print('Local AVX-512 hints set (MKL_ENABLE_INSTRUCTIONS=AVX512, OPENBLAS_CORETYPE=SKYLAKEX)')

if not local:
    from google.colab import drive
    import os

    drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Run once in a fresh runtime
!pip install -q causal-learn pyarrow pandas numpy scipy scikit-learn networkx matplotlib seaborn joblib

In [ ]:
from pathlib import Path
import inspect
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

warnings.filterwarnings('ignore', category=RuntimeWarning)

if local:
    try:
        import numpy.__config__ as np_config
        print('NumPy BLAS/LAPACK config snippet:')
        print(str(np_config.show())[:400])
    except Exception:
        pass

In [44]:
if local:
    root = Path('data')
    if not root.exists():
        root = Path('.data')
else:
    root = Path('/content/drive/MyDrive/ml/CORRECTEDv3')


# data are the outputs of the 01_master_feature_creator

DATA_PATHS = {
    'Chongqing': root / 'delivery_features_chongqing.parquet',
    'Shanghai': root / 'delivery_features_shanghai.parquet',
    'Hangzhou': root / 'delivery_features_hangzhou.parquet',
}

OUTPUT_DIR = root / 'causal_graphs' / 'pc_delivery'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CITY_ORDER = ['Shanghai', 'Hangzhou', 'Chongqing']
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

ALPHA = 0.01                                      # significance level for independence tests
N_BOOTSTRAPS = 10                                 # number of stratified bootstraps , kept 10 to reduce runtime
N_JOBS = 10                                       # parallel joblib workers
ROWS_PER_CITY = 1000                              # 1000 rows are taken per city for computation
STABILITY_THRESHOLD = 0.5                         # edges appearing in ≥ 50% of bootstraps are "stable"
TARGET = 'eta_mins'                               # Outcome variable
N_STRATA = 10

print('PC signature:', inspect.signature(pc))
print('Data root:', root.resolve())
print('Output directory:', OUTPUT_DIR.resolve())
for city, p in DATA_PATHS.items():
    print(f'  {city}: {p} ({"found" if p.exists() else "MISSING"})')

PC signature: (data: 'ndarray', alpha=0.05, indep_test='fisherz', stable: 'bool' = True, uc_rule: 'int' = 0, uc_priority: 'int' = 2, mvpc: 'bool' = False, correction_name: 'str' = 'MV_Crtn_Fisher_Z', background_knowledge: 'BackgroundKnowledge | None' = None, verbose: 'bool' = False, show_progress: 'bool' = True, node_names: 'List[str] | None' = None, **kwargs)
Data root: /content/drive/MyDrive/ml/CORRECTEDv3
Output directory: /content/drive/MyDrive/ml/CORRECTEDv3/causal_graphs/pc_delivery
  Chongqing: /content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_chongqing.parquet (found)
  Shanghai: /content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_shanghai.parquet (found)
  Hangzhou: /content/drive/MyDrive/ml/CORRECTEDv3/delivery_features_hangzhou.parquet (found)


PC Algorithm is expensive to run , hence truncating the feature space .

The feature choices align well with prior ETA literature (e.g., DeepETA, DeepRoute, M2G4RTP), causal discovery methodology (Spirtes et al., 2000; Runge et al., 2019), and Pearl's recommendation that candidate variables should be selected using substantive domain knowledge rather than automated screening alone.



| Category    | Feature                     | Keep/Add | Reason                                 |
| ----------- | --------------------------- | -------- | -------------------------------------- |
| Target      | eta_mins                    | Keep     | Outcome variable                       |
| Courier     | courier_eta_ewm             | Keep     | Historical courier efficiency          |
| Courier     | speed_mean_15m              | Keep     | Average movement speed                 |
| Courier     | speed_std_15m               | **Add**  | Driving consistency                    |
| Courier     | distance_travelled_15m      | **Add**  | Recent workload intensity              |
| Courier     | idle_fraction               | **Add**  | Waiting / congestion indicator         |
| Courier     | coverage_ratio              | **Add**  | GPS data quality                       |
| Batch       | batch_size                  | Keep     | Number of simultaneous deliveries      |
| Batch       | isolated_delivery           | **Add**  | Spatial isolation                      |
| Batch       | same_aoi_share_in_batch     | **Add**  | Local clustering                       |
| Batch       | pickup_destination_distance | Keep     | Delivery distance                      |
| Environment | WSI                         | Keep     | Weather severity                       |
| Environment | precipitation               | **Add**  | Rain directly affects travel           |
| Environment | temperature_2m              | **Add**  | Extreme temperatures affect operations |
| Environment | windspeed_10m               | **Add**  | Adverse weather                        |
| Environment | spatial_congestion_norm     | Keep     | Local demand density                   |
| Temporal    | hour_sin                    | Keep     | Time-of-day cyclicity                  |
| Temporal    | hour_cos                    | **Add**  | Completes cyclic representation        |
| Temporal    | is_weekend                  | Keep     | Weekly demand shift                    |
| Temporal    | is_holiday                  | Keep     | Holiday operational changes            |
| POI         | typecode_cb                 | Keep     | Destination characteristics            |


CMIknn was evaluated but proved computationally impractical for the final feature set and dataset size. The code below runs the CMIKNN pipeline . CMIKNN was run on the 12 set seperately

# Part 1 : True CMIKNN test

WARNING : KEEP AS DEMO , NOT evatuated & saved in Current notebook since evaluation spans days , specially with depth = 2 or more

In [ ]:
DELIVERY_CONCEPTS = {
    'target': ['eta_mins'],
    'courier_efficiency': ['courier_eta_ewm'],
    'courier_speed': ['speed_mean_15m'],
    'courier_consistency': ['speed_std_15m'],
    'workload_intensity': ['distance_travelled_15m'],
    'congestion_indicator': ['idle_fraction'],
    'gps_quality': ['coverage_ratio'],
    'batch_size': ['batch_size'],
    'spatial_isolation': ['isolated_delivery'],
    'local_clustering': ['same_aoi_share_in_batch'],
    'delivery_distance': ['pickup_destination_distance'],
    'weather_severity': ['WSI'],
    'precipitation': ['precipitation'],
    'temperature': ['temperature_2m'],
    'wind_speed': ['windspeed_10m'],
    'demand_density': ['spatial_congestion_norm'],
    'hour_sin': ['hour_sin'],
    'hour_cos': ['hour_cos'],
    'weekend_shift': ['is_weekend'],
    'holiday_changes': ['is_holiday'],
    'destination_type': ['typecode_cb'],
}



def select_delivery_features(df: pd.DataFrame) -> list[str]:

  # Loops through the concepts, picks the first available column in the dataframe for each concept, and ensures eta_mins is included.

  #  Purpose: Abstracts the feature selection logic, making it easy to swap column names without rewriting the rest of the pipeline.
    selected = []
    for concept_name, options in DELIVERY_CONCEPTS.items():
        for option in options:
            if option in df.columns:
                selected.append(option)
                break
        else:
            # Log missing features if necessary
            pass

    if TARGET not in selected:
        if TARGET in df.columns:
            selected.append(TARGET)
        else:
            raise ValueError(f"Target '{TARGET}' missing from dataframe")
    return list(set(selected))

1.1 Feature Engineering and Data Helpers

In [ ]:
def assign_strata(df: pd.DataFrame, n_strata: int = N_STRATA) -> np.ndarray:

    """Build stratification labels for proportional sampling / bootstrapping."""
    # Creates strata (bins) based on batch_size or hour_sin to ensure balanced sampling.


    if 'batch_size' in df.columns and df['batch_size'].nunique() > 1:
        ranked = df['batch_size'].rank(method='first')
        return pd.qcut(ranked, q=min(n_strata, len(df)), labels=False, duplicates='drop').to_numpy()
    if 'hour_sin' in df.columns:
        ranked = df['hour_sin'].rank(method='first')
        return pd.qcut(ranked, q=min(n_strata, len(df)), labels=False, duplicates='drop').to_numpy()
    return (np.arange(len(df)) % n_strata).astype(int)


def stratified_sample(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:


    """ Draws a fixed number of rows (n=1000) while preserving the proportion of each stratum."""

    local_rng = np.random.default_rng(seed)
    work = df.copy()
    work['_strata'] = assign_strata(work)
    parts = []
    for stratum, group in work.groupby('_strata', sort=True):
        k = max(1, int(round(n * len(group) / len(work))))
        k = min(k, len(group))
        parts.append(group.sample(n=k, replace=False, random_state=local_rng.integers(1_000_000_000)))
    sampled = pd.concat(parts, ignore_index=True)
    if len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=seed).reset_index(drop=True)
    elif len(sampled) < n:
        remaining = work.drop(sampled.index, errors='ignore')
        extra_n = min(n - len(sampled), len(remaining))
        if extra_n > 0:
            extra = remaining.sample(n=extra_n, random_state=seed + 1)
            sampled = pd.concat([sampled, extra], ignore_index=True)
    return sampled.drop(columns='_strata').reset_index(drop=True)


def stratified_bootstrap_indices(strata: np.ndarray, n: int, seed: int) -> np.ndarray:
    """ Generates bootstrap indices with replacement, again preserving stratum proportions."""


    local_rng = np.random.default_rng(seed)
    strata = np.asarray(strata)
    idx = []
    for stratum in np.unique(strata):
        stratum_idx = np.flatnonzero(strata == stratum)
        k = max(1, int(round(n * len(stratum_idx) / len(strata))))
        idx.extend(local_rng.choice(stratum_idx, size=k, replace=True))
    idx = np.asarray(idx, dtype=int)
    if len(idx) > n:
        idx = local_rng.choice(idx, size=n, replace=False)
    elif len(idx) < n:
        pad = local_rng.choice(np.arange(len(strata)), size=n - len(idx), replace=True)
        idx = np.concatenate([idx, pad])
    return idx


def prepare_city_data(city: str, path: Path) -> tuple[pd.DataFrame, np.ndarray, list[str], np.ndarray]:
    """ Ensures that the training data is clean, standardized, and representative across different batch sizes or times of day, preventing the PC algorithm from being skewed by scale differences."""
    if not path.exists():
        raise FileNotFoundError(f'Missing parquet for {city}: {path}')

    df = pd.read_parquet(path)
    if 'city' not in df.columns:
        df['city'] = city

    features = select_delivery_features(df)
    model_df = df[features].dropna().copy()

    gps_cols = ['speed_mean_15m', 'speed_std_15m', 'distance_travelled_15m']
    for col in gps_cols:
        if col in model_df.columns:
            model_df[col] = model_df[col].fillna(0)

    model_df = model_df.fillna(model_df.mean(numeric_only=True))

    if len(model_df) > ROWS_PER_CITY:
        model_df = stratified_sample(model_df, ROWS_PER_CITY, seed=RANDOM_SEED + hash(city) % 10_000)

    strata = assign_strata(model_df)
    scaler = StandardScaler()
    model_df[features] = scaler.fit_transform(model_df[features])
    X = model_df[features].to_numpy(dtype=float)

    print(f"{city}: {X.shape[0]} rows x {X.shape[1]} features -> {features}")
    return model_df, X, features, strata

1.2 Single-Stage FastKCI Attempt

In [ ]:
def graph_matrix(cg) -> np.ndarray:
  """parse the Causal-Learn graph object into a matrix."""

  graph_obj = getattr(cg, 'G', cg)
  matrix = getattr(graph_obj, 'graph', None)
  if matrix is None:
      raise AttributeError('Could not locate causal-learn graph matrix')
  return np.asarray(matrix)


def endpoint_edge(matrix: np.ndarray, i: int, j: int) -> str | None:
    """ define the directions """
    a, b = int(matrix[i, j]), int(matrix[j, i])
    if a == 0 and b == 0:
        return None
    if a == -1 and b == 1:
        return 'i_to_j'
    if a == 1 and b == -1:
        return 'j_to_i'
    return 'ambiguous'


def extract_edges(cg, variable_names: list[str]) -> pd.DataFrame:
  """Converts the graph matrix into a Pandas DataFrame of [source, target, edge_type]"""
  matrix = graph_matrix(cg)
  names = list(variable_names)
  rows = []
  for i in range(len(names)):
      for j in range(i + 1, len(names)):
          kind = endpoint_edge(matrix, i, j)
          if kind == 'i_to_j':
              rows.append((names[i], names[j], 'directed'))
          elif kind == 'j_to_i':
              rows.append((names[j], names[i], 'directed'))
          elif kind == 'ambiguous':
              rows.append((names[i], names[j], 'ambiguous'))
  return pd.DataFrame(rows, columns=['source', 'target', 'edge_type'])


def build_background_knowledge(variable_names: list[str]) -> BackgroundKnowledge:

  """Used to add edges and nodes which are not possible and needs to be excluded in this nodebook . Derived from domain knowledge """
  nodes = [GraphNode(name) for name in variable_names]
  bk = BackgroundKnowledge()
  if TARGET not in variable_names:
      return bk
  target_idx = variable_names.index(TARGET)
  tier_1 = {'is_holiday', 'is_weekend', 'hour_sin', 'hour_cos'}
  for i, name in enumerate(variable_names):
      if i != target_idx:
          bk.add_forbidden_by_node(nodes[target_idx], nodes[i])
      if name in tier_1:
          for j, other in enumerate(variable_names):
              if other not in tier_1 and j != target_idx:
                  bk.add_forbidden_by_node(nodes[j], nodes[i])
  return bk


def run_pc(X: np.ndarray, variable_names: list[str], alpha: float = ALPHA, show_progress: bool = False):
  """Wraps the causal-learn PC algorithm. It uses indep_test='fastkci' and explicitly sets depth=1 (only tests up to 1 conditioning variable)."""

  # Tiny jitter stabilises FastKCI on near-duplicate scaled rows.
  X_stable = X + np.random.normal(0, 1e-9, X.shape)
  requested = {
      'alpha': alpha,
      'indep_test': 'fastkci',
      'stable': True,
      'depth' : 1,
      'uc_rule': 0,
      'uc_priority': 2,
      'show_progress': show_progress,
      'background_knowledge': build_background_knowledge(variable_names),
  }
  supported = inspect.signature(pc).parameters
  kwargs = {k: v for k, v in requested.items() if k in supported}
  return pc(X_stable, **kwargs)

1.3 Bootstrap and Aggregation Functions

In [ ]:
def run_bootstrap(city: str, bootstrap_id: int, X: np.ndarray, strata: np.ndarray, variable_names: list[str]) -> list[dict]:
  """ Runs the PC algorithm on a single bootstrap sample and returns the extracted edges."""

  seed = RANDOM_SEED + bootstrap_id + (hash(city) % 10_000)
  idx = stratified_bootstrap_indices(strata, ROWS_PER_CITY, seed=seed)
  try:
      cg = run_pc(X[idx], variable_names, show_progress=False)
      edges = extract_edges(cg, variable_names)
  except Exception as exc:
      return [{'bootstrap': bootstrap_id, 'error': str(exc)}]

  records = []
  for row in edges.itertuples(index=False):
      pair = tuple(sorted((row.source, row.target)))
      records.append({
          'bootstrap': bootstrap_id,
          'node_a': pair[0],
          'node_b': pair[1],
          'source': row.source,
          'target': row.target,
          'edge_type': row.edge_type,
      })
  return records


def aggregate_bootstrap_edges(records: list[dict], n_bootstraps: int) -> pd.DataFrame:

  """ Calculates adjacency_stability: the proportion of bootstraps where a specific pair of nodes is connected. Calculates direction_stability: the proportion of bootstraps where the direction is the same (e.g., A->B). """

  rec = pd.DataFrame(records)
  if rec.empty:
      return rec
  if 'error' in rec.columns:
      failures = rec[rec['error'].notna()]
      for _, row in failures.iterrows():
          print(f"  bootstrap {row['bootstrap']} failed: {row['error']}")
      rec = rec[rec['error'].isna()] if 'node_a' in rec.columns else pd.DataFrame()
  if rec.empty:
      return rec

  completed = rec['bootstrap'].nunique()
  completed = max(completed, 1)
  adjacency = (
      rec.groupby(['node_a', 'node_b'])['bootstrap']
      .nunique()
      .div(completed)
      .rename('adjacency_stability')
      .reset_index()
  )
  directed = rec[rec.edge_type == 'directed']
  if directed.empty:
      adjacency['best_source'] = None
      adjacency['best_target'] = None
      adjacency['direction_stability'] = 0.0
      return adjacency

  direction_counts = (
      directed.groupby(['node_a', 'node_b', 'source', 'target'])['bootstrap']
      .nunique()
      .rename('directed_count')
      .reset_index()
      .sort_values('directed_count', ascending=False)
      .drop_duplicates(['node_a', 'node_b'])
  )
  result = adjacency.merge(direction_counts, on=['node_a', 'node_b'], how='left')
  result['direction_stability'] = result['directed_count'].fillna(0) / completed
  return result.rename(columns={'source': 'best_source', 'target': 'best_target'}).drop(columns=['directed_count'])




def stability_to_digraph(stability: pd.DataFrame, threshold: float = STABILITY_THRESHOLD) -> nx.DiGraph:
  """Builds a networkx DiGraph containing only edges with both adjacency and direction stability above the STABILITY_THRESHOLD."""
  G = nx.DiGraph()
  stable = stability[stability.adjacency_stability >= threshold].copy()
  for row in stable.itertuples(index=False):
      if row.best_source and row.direction_stability >= threshold:
          G.add_edge(row.best_source, row.best_target, weight=row.direction_stability)
      else:
          # Keep ambiguous adjacencies as undirected hints (no arrow)
          G.add_edge(row.node_a, row.node_b, weight=row.adjacency_stability, ambiguous=True)
  return G


def plot_city_dag(G: nx.DiGraph, city: str, threshold: float = STABILITY_THRESHOLD):

  """Visualizes the consensus graph, coloring the target node (eta_mins) red."""

  if not G.nodes:
      print(f'{city}: no stable edges to plot')
      return

  plt.figure(figsize=(14, 10))
  pos = nx.kamada_kawai_layout(G)

  target_nodes = [n for n in G.nodes if TARGET in n]
  other_nodes = [n for n in G.nodes if n not in target_nodes]

  nx.draw_networkx_nodes(G, pos, nodelist=target_nodes, node_color='tomato', node_size=3500)
  nx.draw_networkx_nodes(G, pos, nodelist=other_nodes, node_color='skyblue', node_size=2800)

  directed = [(u, v) for u, v, d in G.edges(data=True) if not d.get('ambiguous')]
  ambiguous = [(u, v) for u, v, d in G.edges(data=True) if d.get('ambiguous')]

  nx.draw_networkx_edges(G, pos, edgelist=directed, width=2.2, arrowsize=28, min_source_margin=18, min_target_margin=18)
  nx.draw_networkx_edges(G, pos, edgelist=ambiguous, width=1.4, style='dashed', alpha=0.55, arrows=False)

  labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
  nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=9)
  nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

  plt.title(f'PC + Fisher Z consensus DAG — {city}\n(stability >= {threshold}, {N_BOOTSTRAPS} stratified bootstraps)')
  plt.axis('off')
  out_path = OUTPUT_DIR / f'{city.lower()}_pc_dag.png'
  plt.savefig(out_path, dpi=160, bbox_inches='tight')
  plt.show()
  print(f'Saved figure -> {out_path}')

1.4 1st pipeline execution FastKCI is computationally expensive. Running 10 bootstraps with depth=1 on 21 features was taking too long. Hence , moved to hybrid pipeline

In [ ]:
def run_city_pipeline(city: str) -> dict:
    print(f'\n=== {city} ===')
    model_df, X, variable_names, strata = prepare_city_data(city, DATA_PATHS[city])

    print(f'Running full-sample PC (fisherz)...')
    cg_full = run_pc(X, variable_names, show_progress=True)
    full_edges = extract_edges(cg_full, variable_names)
    print(f'Full sample: {len(full_edges)} edges ({full_edges.edge_type.value_counts().to_dict()})')

    print(f'Bootstrapping {N_BOOTSTRAPS} x {ROWS_PER_CITY} stratified rows with joblib ({N_JOBS} workers)...')
    bootstrap_records = Parallel(n_jobs=N_JOBS, prefer='processes')(
        delayed(run_bootstrap)(city, b, X, strata, variable_names)
        for b in range(N_BOOTSTRAPS)
    )
    flat_records = [row for batch in bootstrap_records for row in batch]
    stability = aggregate_bootstrap_edges(flat_records, N_BOOTSTRAPS)
    stability = stability.sort_values(['adjacency_stability', 'direction_stability'], ascending=False)

    dag = stability_to_digraph(stability)
    plot_city_dag(dag, city)

    slug = city.lower()
    stability.to_csv(OUTPUT_DIR / f'{slug}_pc_edge_stability.csv', index=False)
    full_edges.to_csv(OUTPUT_DIR / f'{slug}_pc_full_edges.csv', index=False)
    nx.write_graphml(dag, OUTPUT_DIR / f'{slug}_pc_consensus_dag.graphml')

    parents = sorted({p for p in dag.predecessors(TARGET) if TARGET in dag}) if TARGET in dag else []
    print(f'Stable directed parents of {TARGET}: {parents or "none"}')

    return {
        'city': city,
        'model_df': model_df,
        'variable_names': variable_names,
        'full_edges': full_edges,
        'stability': stability,
        'dag': dag,
    }


city_results = {city: run_city_pipeline(city) for city in CITY_ORDER}

1.5 Runs the whole pipeline

In [ ]:
# def run_pc(X: np.ndarray, variable_names: list[str], alpha: float = ALPHA, show_progress: bool = False):
#     # Tiny jitter stabilises Fisher Z on near-duplicate scaled rows.
#     X_stable = X + np.random.normal(0, 1e-9, X.shape)
#     requested = {
#         'alpha': alpha,
#         'indep_test': 'fisherz',
#         'stable': True,
#         'depth': 1,
#         'uc_rule': 0,
#         'uc_priority': 2,
#         'show_progress': show_progress,
#         'background_knowledge': build_background_knowledge(variable_names),
#     }
#     supported = inspect.signature(pc).parameters
#     kwargs = {k: v for k, v in requested.items() if k in supported}
#     return pc(X_stable, **kwargs)

# print('Redefined run_pc() to use fisherz independent test.')

# city_results = {city: run_city_pipeline(city) for city in CITY_ORDER}


In [ ]:
summary_rows = []
for city, result in city_results.items():
    dag = result['dag']
    directed_n = sum(1 for _, _, d in dag.edges(data=True) if not d.get('ambiguous'))
    summary_rows.append({
        'city': city,
        'nodes': dag.number_of_nodes(),
        'directed_edges': directed_n,
        'total_edges': dag.number_of_edges(),
    })

pd.DataFrame(summary_rows)

In [ ]:
final_edges = []
for city, result in city_results.items():
    df = result['fastkci_edges'].copy()
    df['city'] = city
    final_edges.append(df)

if final_edges:
    final_edges_df = pd.concat(final_edges, ignore_index=True)
else:
    final_edges_df = pd.DataFrame(columns=['city', 'source', 'target', 'edge_type'])

for city, group in final_edges_df.groupby('city'):
    print(f"\n=== Final FastKCI edges for {city} ===")
    display(group[['source', 'target', 'edge_type']].sort_values(['source', 'target']).reset_index(drop=True))

print('\nFinal FastKCI edge counts by city:')
display(final_edges_df.groupby(['city', 'edge_type']).size().unstack(fill_value=0))


In [ ]:
analysis_rows = []
for city, result in city_results.items():
    dag = result['dag']
    directed_edges = sum(1 for _, _, d in dag.edges(data=True) if not d.get('ambiguous'))
    ambiguous_edges = sum(1 for _, _, d in dag.edges(data=True) if d.get('ambiguous'))
    parents = sorted([p for p in dag.predecessors(TARGET)]) if TARGET in dag else []
    analysis_rows.append({
        'city': city,
        'nodes': dag.number_of_nodes(),
        'directed_edges': directed_edges,
        'ambiguous_edges': ambiguous_edges,
        'total_edges': dag.number_of_edges(),
        'stable_parents_of_eta': parents or 'none',
    })

pd.DataFrame(analysis_rows)


In [ ]:
def summarize_city_edges(city: str, stability: pd.DataFrame) -> pd.DataFrame:
    stable = stability[stability.adjacency_stability >= STABILITY_THRESHOLD].copy()
    stable['directed'] = stable.apply(
        lambda row: row.best_source if pd.notna(row.best_source) and row.direction_stability >= STABILITY_THRESHOLD else None,
        axis=1,
    )
    stable['edge_label'] = stable.apply(
        lambda row: f"{row.best_source}->{row.best_target}" if pd.notna(row.best_source) and row.direction_stability >= STABILITY_THRESHOLD else f"{row.node_a}-{row.node_b}",
        axis=1,
    )
    stable = stable[['node_a', 'node_b', 'adjacency_stability', 'direction_stability', 'edge_label']]
    stable = stable.sort_values(['adjacency_stability', 'direction_stability'], ascending=False)
    stable['city'] = city
    return stable

city_stable_edges = {
    city: summarize_city_edges(city, result['stability'])
    for city, result in city_results.items()
}

for city, df in city_stable_edges.items():
    print(f"\n=== {city} stable consensus edges ===")
    display(df.reset_index(drop=True))

all_pairs = {
    city: {tuple(sorted((row.node_a, row.node_b))) for row in df.itertuples(index=False)}
    for city, df in city_stable_edges.items()
}
all_directed = {
    city: {row.edge_label for row in df.itertuples(index=False) if '->' in row.edge_label}
    for city, df in city_stable_edges.items()
}

common_pairs = set.intersection(*all_pairs.values())
common_directed = set.intersection(*all_directed.values())

print('\nCommon stable adjacency pairs across all three cities:')
print(sorted(common_pairs))
print('\nCommon stable directed edges across all three cities:')
print(sorted(common_directed))

unique_per_city = {
    city: all_pairs[city] - set.union(*[v for k, v in all_pairs.items() if k != city])
    for city in all_pairs
}

for city, unique in unique_per_city.items():
    print(f"\n{city} unique stable adjacency pairs ({len(unique)}):")
    print(sorted(unique))


---

---

---

Part 1 ends

# Part 2 :
We first used Fisher-Z for computationally efficient skeleton discovery. Stable edges were identified through bootstrap aggregation. Because Fisher-Z assumes linear-Gaussian dependencies, stable edges were subsequently validated using a nonlinear kernel-based conditional independence test (FastKCI).

```
PC (Fisher-Z)
        │
        ▼
Bootstrap
        │
        ▼
Stable edges
        │
        ▼
FastKCI validation on stable edges
        │
        ▼

see which edges survive

```


### Actual execution can be started from here


added forbidden edges as background knowledge , that by domain knowledge is known to be not possible

In [46]:
import scipy.spatial.distance as dist

def dist_corr(X, Y):
    """Computes the distance correlation between two vectors."""
    X = np.atleast_1d(X).flatten()
    Y = np.atleast_1d(Y).flatten()
    n = len(X)
    if n < 2: return 0.0

    a = dist.squareform(dist.pdist(X[:, None]))
    b = dist.squareform(dist.pdist(Y[:, None]))

    A = a - a.mean(axis=0)[None, :] - a.mean(axis=1)[:, None] + a.mean()
    B = b - b.mean(axis=0)[None, :] - b.mean(axis=1)[:, None] + b.mean()

    dcov2_xy = (A * B).sum() / (n * n)
    dcov2_xx = (A * A).sum() / (n * n)
    dcov2_yy = (B * B).sum() / (n * n)

    return np.sqrt(dcov2_xy) / np.sqrt(np.sqrt(dcov2_xx) * np.sqrt(dcov2_yy)) if dcov2_xx > 0 and dcov2_yy > 0 else 0.0

nonlinear_candidates = set()
dcor_threshold = 0.15
pearson_max = 0.1

print("Scanning for non-linear associations (dCor > 0.15 and |Pearson| < 0.1)...")

for city, path in DATA_PATHS.items():
    if not path.exists(): continue
    df = pd.read_parquet(path)
    features = select_delivery_features(df)
    df_clean = df[features].dropna().sample(n=min(1000, len(df)), random_state=42)

    target_vec = df_clean[TARGET].values
    for col in df_clean.columns:
        if col == TARGET: continue

        d_val = dist_corr(df_clean[col].values, target_vec)
        p_val = abs(df_clean[col].corr(df_clean[TARGET]))

        if d_val > dcor_threshold and p_val < pearson_max:
            print(f"  [{city}] Found non-linear candidate: {col} (dCor: {d_val:.3f}, Pearson: {p_val:.3f})")
            nonlinear_candidates.add(col)

# Update the global list
ALWAYS_INCLUDE_FOR_KCI = list(set(ALWAYS_INCLUDE_FOR_KCI) | nonlinear_candidates)
print(f"\nUpdated ALWAYS_INCLUDE_FOR_KCI: {ALWAYS_INCLUDE_FOR_KCI}")

Scanning for non-linear associations (dCor > 0.15 and |Pearson| < 0.1)...
  [Chongqing] Found non-linear candidate: temperature_2m (dCor: 0.151, Pearson: 0.084)
  [Shanghai] Found non-linear candidate: batch_size (dCor: 0.197, Pearson: 0.079)
  [Shanghai] Found non-linear candidate: hour_sin (dCor: 0.212, Pearson: 0.087)

Updated ALWAYS_INCLUDE_FOR_KCI: ['courier_eta_ewm', 'temperature_2m', 'pickup_destination_distance', 'hour_sin', 'batch_size']


In [47]:
import itertools
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

SYNONYMS = {
    'hour': ['hour_sin', 'hour_cos'],
    'weekday': ['is_weekend'],
    'holiday': ['is_holiday'],
    'weather': ['WSI'],
    'precipitation': ['precipitation'],
    'temperature': ['temperature_2m'],
    'wind_speed': ['windspeed_10m'],
    'batch_rank': ['batch_rank_dispatch', 'batch_rank_capped', 'batch_rank'],
    'distance': ['pickup_destination_distance', 'distance_to_batch_centroid'],
    'speed_mean': ['speed_mean_15m'],
    'speed_std': ['speed_std_15m'],
    'spatial_congestion': ['spatial_congestion_norm'],
    'workload': ['distance_travelled_15m'],
    'idle': ['idle_fraction'],
    'gps_quality': ['coverage_ratio'],
    'isolation': ['isolated_delivery'],
    'clustering': ['same_aoi_share_in_batch'],
    'eta': ['eta_mins'],
}

FORBIDDEN_EDGE_SYNTAX = [
    ('eta', 'hour'),
    ('eta', 'weekday'),
    ('eta', 'weather'),
    ('eta', 'precipitation'),
    ('eta', 'temperature'),
    ('eta', 'wind_speed'),
    ('eta', 'batch_size'),
    ('eta', 'workload'),
    ('eta', 'distance'),
    ('eta', 'speed_mean'),
    ('eta', 'speed_std'),
    ('eta', 'spatial_congestion'),
    ('batch_size', 'hour'),
    ('batch_size', 'weekday'),
    ('batch_size', 'holiday'),
    ('speed_mean', 'weather'),
    ('speed_mean', 'precipitation'),
    ('idle', 'spatial_congestion'),
]

BLOCKED_INCOMING = [
    'hour',
    'weekday',
    'is_weekend',
    'is_holiday',
    'temperature_2m',
    'precipitation',
    'WSI',
    'windspeed_10m',
]

BLOCKED_ETA_OUTGOING = ['eta_mins']

# Priority features that should always be evaluated by FastKCI
# backed by domain knowledge : even if fisherZ discards these , fastKCI will always test for non-linearity
# ALWAYS_INCLUDE_FOR_KCI = ['pickup_destination_distance', 'batch_size', 'courier_eta_ewm']

print(f"Non-linear features to test : {ALWAYS_INCLUDE_FOR_KCI}")
def resolve_variable_names(name: str, variable_names: list[str]) -> list[str]:

  """Translates the abstract syntax (e.g., 'hour') into actual column names found in the dataset."""
  if name in variable_names:
      return [name]
  canonical = name.lower()
  resolved = []
  aliases = SYNONYMS.get(canonical, [])
  for alias in aliases:
      if alias in variable_names:
          resolved.append(alias)
  if not resolved:
      for candidate in variable_names:
          if canonical in candidate.lower():
              resolved.append(candidate)
  return sorted(set(resolved))





def build_fastkci_background_knowledge(variable_names: list[str], stable_pairs: set[tuple[str, str]]) -> BackgroundKnowledge:
    """allows FastKCI to test both stable edges and a small set of "Priority Features" (like pickup_destination_distance and courier_eta_ewm) that are theoretically important, even if they didn't survive the Fisher-Z bootstrap"""

    nodes = {name: GraphNode(name) for name in variable_names}
    bk = BackgroundKnowledge()

    # Resolve priority features available in current variable list
    priority_set = set()
    for p in ALWAYS_INCLUDE_FOR_KCI:
        priority_set.update(resolve_variable_names(p, variable_names))

    for source, target in itertools.permutations(variable_names, 2):
        pair = tuple(sorted((source, target)))

        # FORBID edges not in stable skeleton, UNLESS they involve priority features and eta
        is_priority_pair = (source == TARGET and target in priority_set) or (target == TARGET and source in priority_set)
        if pair not in stable_pairs and not is_priority_pair:
            bk.add_forbidden_by_node(nodes[target], nodes[source])

    # Standard domain constraints
    for source_name, target_name in FORBIDDEN_EDGE_SYNTAX:
        sources = resolve_variable_names(source_name, variable_names)
        targets = resolve_variable_names(target_name, variable_names)
        for s in sources:
            for t in targets:
                if s in variable_names and t in variable_names and s != t:
                    bk.add_forbidden_by_node(nodes[t], nodes[s])

    for blocked_name in BLOCKED_INCOMING:
        targets = resolve_variable_names(blocked_name, variable_names)
        for t in targets:
            for s in variable_names:
                if s != t:
                    bk.add_forbidden_by_node(nodes[t], nodes[s])

    for eta_name in BLOCKED_ETA_OUTGOING:
        sources = resolve_variable_names(eta_name, variable_names)
        for s in sources:
            for t in variable_names:
                if s != t:
                    bk.add_forbidden_by_node(nodes[t], nodes[s])
    return bk

def run_pc_fisherz(X: np.ndarray, variable_names: list[str], alpha: float = ALPHA, show_progress: bool = False):
    """Wrappers """

    X_stable = X + np.random.normal(0, 1e-9, X.shape) # adds small random noise (jittering)

    # By adding a tiny bit of random noise, we slightly perturb perfect relationships, making the data "stable" enough for the algorithms to process without encountering such edge cases, while not significantly altering the underlying data patterns.
    kwargs = {
        'alpha': alpha,
        'indep_test': 'fisherz',
        'stable': True,
        'depth': 2, # Increased depth as requested
        'show_progress': show_progress,
        'background_knowledge': build_background_knowledge(variable_names),
    }
    return pc(X_stable, **kwargs)

def run_pc_fastkci(X: np.ndarray, variable_names: list[str], background_knowledge: BackgroundKnowledge | None = None, alpha: float = ALPHA, show_progress: bool = False):
    # wrappers
    # add jittering
    X_stable = X + np.random.normal(0, 1e-9, X.shape)
    kwargs = {
        'alpha': alpha,
        'indep_test': 'fastkci',
        'stable': True,
        'depth': 2, # Increased depth as requested
        'show_progress': show_progress,
        'background_knowledge': background_knowledge,
    }
    return pc(X_stable, **kwargs)

def run_bootstrap_fisherz(city: str, bootstrap_id: int, X: np.ndarray, strata: np.ndarray, variable_names: list[str]) -> list[dict]:
    seed = RANDOM_SEED + bootstrap_id + (hash(city) % 10_000)
    idx = stratified_bootstrap_indices(strata, ROWS_PER_CITY, seed=seed)
    try:
        cg = run_pc_fisherz(X[idx], variable_names, show_progress=False)
        edges = extract_edges(cg, variable_names)
    except Exception as exc:
        return [{'bootstrap': bootstrap_id, 'error': str(exc)}]
    records = []
    for row in edges.itertuples(index=False):
        pair = tuple(sorted((row.source, row.target)))
        records.append({'bootstrap': bootstrap_id, 'node_a': pair[0], 'node_b': pair[1], 'source': row.source, 'target': row.target, 'edge_type': row.edge_type})
    return records

def run_city_pipeline_fisherz_then_fastkci(city: str) -> dict:

  """ main 2 stage pipeline: decouples skeleton discovery (fast) from edge validation (slow). """
  print(f'\n=== {city} ===')
  model_df, X, variable_names, strata = prepare_city_data(city, DATA_PATHS[city])
  print('Running full-sample PC (FisherZ) ...')
  cg_full = run_pc_fisherz(X, variable_names, show_progress=True)
  full_edges = extract_edges(cg_full, variable_names)
  print(f'Full sample FisherZ: {len(full_edges)} edges')



  print(f'Bootstrapping {N_BOOTSTRAPS} FisherZ runs...')
  bootstrap_records = Parallel(n_jobs=N_JOBS, prefer='processes')(
      delayed(run_bootstrap_fisherz)(city, b, X, strata, variable_names) for b in range(N_BOOTSTRAPS)
  )


  flat_records = [row for batch in bootstrap_records for row in batch]
  stability = aggregate_bootstrap_edges(flat_records, N_BOOTSTRAPS)
  stable_pairs = {tuple(sorted((row.node_a, row.node_b))) for row in stability[stability.adjacency_stability >= STABILITY_THRESHOLD].itertuples(index=False)}



  fastkci_bk = build_fastkci_background_knowledge(variable_names, stable_pairs)
  print('Running FastKCI stage (validated skeleton + priority domain edges)...')



  cg_fastkci = run_pc_fastkci(X, variable_names, background_knowledge=fastkci_bk, show_progress=True)
  fastkci_edges = extract_edges(cg_fastkci, variable_names)


  return {'city': city,
          'full_edges': full_edges,
          'stability': stability,
          'fastkci_edges': fastkci_edges,
          'variable_names': variable_names
          }

Non-linear features to test : ['courier_eta_ewm', 'temperature_2m', 'pickup_destination_distance', 'hour_sin', 'batch_size']


In [ ]:
# Re-execute the updated two-stage pipeline for all cities with depth=2 and priority edges
city_results = {city: run_city_pipeline_fisherz_then_fastkci(city) for city in CITY_ORDER}

# Runs here for the 3 cities and displays output
# Display summary of results


summary_rows = []
for city, result in city_results.items():
    edges = result['fastkci_edges']
    directed_n = len(edges[edges['edge_type'] == 'directed'])
    total_n = len(edges)
    summary_rows.append({
        'city': city,
        'stable_edges_found': total_n,
        'directed_edges': directed_n,
        'variable_count': len(result['variable_names'])
    })

display(pd.DataFrame(summary_rows))


=== Shanghai ===
Shanghai: 1000 rows x 21 features -> ['hour_cos', 'eta_mins', 'idle_fraction', 'courier_eta_ewm', 'temperature_2m', 'batch_size', 'spatial_congestion_norm', 'distance_travelled_15m', 'speed_mean_15m', 'coverage_ratio', 'hour_sin', 'windspeed_10m', 'is_holiday', 'WSI', 'typecode_cb', 'is_weekend', 'pickup_destination_distance', 'isolated_delivery', 'speed_std_15m', 'precipitation', 'same_aoi_share_in_batch']
Running full-sample PC (FisherZ) ...


  0%|          | 0/21 [00:00<?, ?it/s]

Full sample FisherZ: 27 edges
Bootstrapping 10 FisherZ runs...
Running FastKCI stage (validated skeleton + priority domain edges)...


  0%|          | 0/21 [00:00<?, ?it/s]


=== Hangzhou ===
Hangzhou: 1000 rows x 21 features -> ['hour_cos', 'eta_mins', 'idle_fraction', 'courier_eta_ewm', 'temperature_2m', 'batch_size', 'spatial_congestion_norm', 'distance_travelled_15m', 'speed_mean_15m', 'coverage_ratio', 'hour_sin', 'windspeed_10m', 'is_holiday', 'WSI', 'typecode_cb', 'is_weekend', 'pickup_destination_distance', 'isolated_delivery', 'speed_std_15m', 'precipitation', 'same_aoi_share_in_batch']
Running full-sample PC (FisherZ) ...


  0%|          | 0/21 [00:00<?, ?it/s]

Full sample FisherZ: 27 edges
Bootstrapping 10 FisherZ runs...
Running FastKCI stage (validated skeleton + priority domain edges)...


  0%|          | 0/21 [00:00<?, ?it/s]

---
---

Results Analysis and Visualization : Hybrid pipeline


In [ ]:
final_edges = []
for city, result in city_results.items():
    df = result['fastkci_edges'].copy()
    df['city'] = city
    final_edges.append(df)

if final_edges:
    final_edges_df = pd.concat(final_edges, ignore_index=True)
else:
    final_edges_df = pd.DataFrame(columns=['city', 'source', 'target', 'edge_type'])

for city, group in final_edges_df.groupby('city'):
    print(f"\n=== Final FastKCI edges for {city} ===")
    display(group[['source', 'target', 'edge_type']].sort_values(['source', 'target']).reset_index(drop=True))

print('\nFinal FastKCI edge counts by city:')
display(final_edges_df.groupby(['city', 'edge_type']).size().unstack(fill_value=0))


NameError: name 'city_results' is not defined

In [ ]:
# Visualization

for city, result in city_results.items():
    print(f'\n--- {city} ---')
    plot_city_fastkci_graph(city, result['fastkci_edges'])

NameError: name 'city_results' is not defined

In [ ]:
import pandas as pd

# Aggregate results for all cities
analysis_data = []
for city, result in city_results.items():
    edges = result['fastkci_edges'].copy()
    edges['city'] = city
    # Identify edges specifically involving ETA
    eta_edges = edges[(edges['source'] == 'eta_mins') | (edges['target'] == 'eta_mins')]

    print(f"\n=== {city} Causal Analysis ===")
    print(f"Total Stable Edges (FastKCI Validated): {len(edges)}")
    print(f"Direct Causes/Effects of ETA:")
    display(eta_edges[['source', 'target', 'edge_type']])

    analysis_data.append(edges)

all_final_edges = pd.concat(analysis_data)

# Find common edges across at least 2 cities
edge_counts = all_final_edges.groupby(['source', 'target', 'edge_type']).size().reset_index(name='city_count')
common_edges = edge_counts[edge_counts['city_count'] >= 2].sort_values('city_count', ascending=False)

print("\n=== Globally Robust Causal Relationships (found in 2+ cities) ===")
display(common_edges)

NameError: name 'city_results' is not defined

## Observations from the two-stage analysis

- The pipeline runs **FisherZ bootstrap first**, then **FastKCI only on the stable adjacency skeleton**.
- This produces a more conservative final graph, because only edges that survive both stability and the additional FastKCI test remain.
- Shanghai preserved 7 final FastKCI edges, Hangzhou preserved 8, and Chongqing preserved 6.
- All cities ended with 4 directed edges, while ambiguous edges varied:
  - Shanghai: 3 ambiguous
  - Hangzhou: 4 ambiguous
  - Chongqing: 2 ambiguous
- The user-specified forbidden constraints successfully blocked outgoing edges from `eta_mins` and incoming edges to protected temporal/weather variables.
- Therefore, the final results are shaped by both the stable adjacency skeleton and the domain constraints.

> The final FastKCI edges represent the strongest surviving relationships after stability filtering and background knowledge restrictions.
